# Week 5, Day 5: Pro RAG on WWE docs: LLM chunks, native search, rerank, rewrite

**Lesson:** [week5/day5.ipynb](../../../week5/day5.ipynb), *Going pro, in three steps*. **About 45 minutes.**

**What you practice here**

- Pre-processing documents with an LLM: a `Chunk` schema with described fields, and one LiteLLM `completion` call per document that asks for `Chunks`, read back with `model_validate_json`.
- Native vector search without LangChain: the question embedded with `client.embeddings.create`, then `collection.query` and its lists of lists.
- Reranking the retrieved chunks with an LLM and a `RankOrder` schema, the chunks numbered from 1 in the prompt.
- Rewriting a follow-up question into a short search query, using the conversation so far.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week5/day5", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson goes pro without LangChain. An LLM cuts each document into chunks with a headline and a summary, the chunks
go into Chroma with OpenAI embeddings, and two more LLM calls improve retrieval: one rewrites the user's question
into a search query, and one reranks the chunks that come back. Build these pieces for your WWE documents.

The functions that call a model take LiteLLM's `completion` function as their first parameter: `chunk_document`,
`rerank` and `rewrite_query`. `retrieve` takes the OpenAI client and a Chroma collection. The checks pass stand-ins
shaped like them, which record what your code sends and answer with canned replies, so nothing is spent. In Try it for
real you store your own pre-processed chunks and ask a follow-up question about Edge.

## What each call should give

```python
document = {"type": "people", "source": "plan/docs/people/edge.md", "text": "# Edge\n\n..."}

chunk_document(completion, document)
# makes one request with the model gpt-4.1-nano, one user message holding your prompt, and Chunks as the response
# format, and returns a Result per chunk, such as
# Result(page_content='Edge wins the first Money in the Bank\n\nEdge won the ...\n\nEdge won the ...',
#        metadata={'source': 'plan/docs/people/edge.md', 'type': 'people'})

retrieve(openai, collection, "When did Edge cash in Money in the Bank?", k=2)
# embeds the question once, asks the collection for its 2 nearest chunks, and returns them as two Results

rerank(completion, "When did Edge cash in Money in the Bank?", chunks)
# the chunks in the model's order: for the reply {"order": [3, 1, 2]}, [chunks[2], chunks[0], chunks[1]]

rewrite_query(completion, "When did he cash it in?", history)
# 'When did Edge cash in his Money in the Bank contract?'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from pydantic import BaseModel, Field

## Given

Run this cell. It holds the lesson's `Result`, which stands in for LangChain's `Document` and is what the parts return, and the lesson's average chunk size, in characters.

In [ ]:
AVERAGE_CHUNK_SIZE = 500


class Result(BaseModel):
    page_content: str
    metadata: dict

### 1 · `class Chunk(BaseModel), class Chunks(BaseModel)`

Describes the chunks the model returns, as the lesson's `Chunk` and `Chunks` do. `Chunk` has three string fields,
`headline`, `summary` and `original_text`, each described to the model in 20 characters or more, saying what to put
there. Give it a method `as_result(self, document)` that returns a `Result` whose text is the headline, the summary
and the original text, with a blank line between each, and whose metadata holds the document's `"source"` and
`"type"`.

Then `Chunks` has one field, `chunks`, a list of `Chunk`: the model's reply must be one object, and this is its shape.

<details><summary>Hint</summary>

```python
headline: str = Field(description="...")

def as_result(self, document):
    metadata = {"source": document["source"], "type": document["type"]}
    return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
                  metadata=metadata)
```

And in `Chunks`, `chunks: list[Chunk]`.

</details>

In [ ]:
class Chunk(BaseModel):
    # Write the three fields and as_result here, as described above.
    pass


class Chunks(BaseModel):
    # Write the one field here, as described above.
    pass

In [ ]:
check("Chunks")

### 2 · `chunk_document(completion, document, model="gpt-4.1-nano")`

Has the model cut one document into chunks, in one call, and returns them as `Result`s. `document` is a dict with a
`"type"`, a `"source"` and a `"text"`, like the lesson's. Write a prompt of your own that gives the document's type,
its source and its full text, and asks the model to split it into overlapping chunks (use the word "overlap"), each
with a headline, a summary and the original text. Say how many chunks to aim for: the text's length divided by
`AVERAGE_CHUNK_SIZE`, rounded down, plus 1, as the lesson does.

Make one call with the model `model`, one user message holding your prompt, and `Chunks` itself as the response
format. Read the reply's JSON back into `Chunks`, and return each chunk's `as_result(document)`, in order.

<details><summary>Hint</summary>

```python
how_many = len(document["text"]) // AVERAGE_CHUNK_SIZE + 1
response = completion(model=model, messages=[{"role": "user", "content": prompt}], response_format=Chunks)
chunks = Chunks.model_validate_json(response.choices[0].message.content).chunks
```

</details>

In [ ]:
def chunk_document(completion, document, model="gpt-4.1-nano"):
    # Write chunk_document here, as described above.
    raise NotImplementedError("chunk_document is not written yet")

In [ ]:
check("chunk_document")

### 3 · `retrieve(client, collection, question, k=10, model="text-embedding-3-large")`

Finds the `k` chunks nearest to the question in a Chroma collection, without LangChain. Make one embeddings call with
the model `model` and a list holding the question, and take the question's vector from the first item of the reply's
data. Then ask the collection for the `k` results nearest that vector. Chroma answers with one list per query vector,
so take the first list of documents and the first list of metadatas, and return a `Result` for each pair, in the
order Chroma gives, nearest first.

<details><summary>Hint</summary>

```python
vector = client.embeddings.create(model=model, input=[question]).data[0].embedding
results = collection.query(query_embeddings=[vector], n_results=k)
pairs = zip(results["documents"][0], results["metadatas"][0])
```

</details>

In [ ]:
def retrieve(client, collection, question, k=10, model="text-embedding-3-large"):
    # Write retrieve here, as described above.
    raise NotImplementedError("retrieve is not written yet")

In [ ]:
check("retrieve")

### 4 · `class RankOrder(BaseModel), rerank(completion, question, chunks, model="gpt-4.1-nano")`

Has the model put the retrieved chunks in order of relevance, in one call. First write `RankOrder`, with one field,
`order`, a list of integers, described to the model in 20 characters or more.

Then `rerank` makes one call with the model `model`, two messages, and `RankOrder` itself as the response format. The
system message, of 20 characters or more, gives the re-ranker its job. The user message holds the question and then,
for each chunk in the order given, `# CHUNK ID: <id>:`, a blank line, and the chunk's text, with ids counted from 1,
as the lesson writes them. Read the ids from the reply, and return the chunks in that order: id 1 is `chunks[0]`.
Skip any id outside 1 to the number of chunks.

<details><summary>Hint</summary>

```python
order: list[int] = Field(description="...")

user += f"# CHUNK ID: {index}:\n\n{chunk.page_content}\n\n"
response = completion(model=model, messages=messages, response_format=RankOrder)
order = RankOrder.model_validate_json(response.choices[0].message.content).order
return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]
```

</details>

In [ ]:
class RankOrder(BaseModel):
    # Write the one field here, as described above.
    pass


def rerank(completion, question, chunks, model="gpt-4.1-nano"):
    # Write rerank here, as described above.
    raise NotImplementedError("rerank is not written yet")

In [ ]:
check("rerank")

### 5 · `rewrite_query(completion, question, history, model="gpt-4.1-nano")`

Turns a follow-up question into a short search query, using the conversation so far, in one call. `history` is a
list of messages, each a dict with a `"role"` and a `"content"`. Make one call with the model `model`, whose messages
show the content of every history message and the current question, and ask for one short, specific query for the
knowledge base, and nothing else; the lesson puts all of this in one system message. Return the reply's text with
the whitespace around it removed.

<details><summary>Hint</summary>

```python
message = f"""... This is the conversation so far:
{history}
And this is the user's current question:
{question}
Respond only with ..."""
response = completion(model=model, messages=[{"role": "system", "content": message}])
return response.choices[0].message.content.strip()
```

</details>

In [ ]:
def rewrite_query(completion, question, history, model="gpt-4.1-nano"):
    # Write rewrite_query here, as described above.
    raise NotImplementedError("rewrite_query is not written yet")

In [ ]:
check("rewrite_query")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 5, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Pre-processing makes one `gpt-4.1-nano` call per document: a few minutes for all of them, and a few cents. Embedding the chunks with `text-embedding-3-large` costs less than a cent.

1. Load your keys with `load_dotenv(override=True)` and make `openai = OpenAI()`.
2. Load your documents with the lesson's homemade loader: for each folder in `DOCS = ROOT / "plan" / "docs"` (the
   setup cell set `ROOT` to your fork's folder) and each `.md` file below it, a dict with the folder's name as `type`,
   the path from the fork's top as `source` (`plan/docs/<folder>/<file>.md`), and the file's text. Then pre-process
   them: collect `chunk_document(completion, document)` for every document, with `completion` from `litellm`.
3. Store the chunks natively, as the lesson's `create_embeddings` does. Open `PersistentClient` on
   `ROOT / "vector_db"`, the folder Day 2 made. Drop its `wwe_pro` collection if it is there, and make it again with
   `get_or_create_collection("wwe_pro")`. Embed all the chunks' texts in one `openai.embeddings.create` call with
   `text-embedding-3-large`, and add them to the collection with their ids, embeddings, texts and metadatas. Keep the
   folder name `vector_db`: the fork's `.gitignore` leaves it out, but not the lesson's `preprocessed_db`.
4. Answer a follow-up. Given a short history about the first Money in the Bank match, `rewrite_query` "When did he
   cash it in?"; `retrieve` and `rerank` with the query; build a system prompt from "Extract from <source>:" blocks
   of the top five chunks, as the lesson's `make_rag_messages` does; and answer with `completion`.

<details><summary>Stuck? One way to write it</summary>

```python
from chromadb import PersistentClient
from dotenv import load_dotenv
from litellm import completion
from openai import OpenAI
from tqdm import tqdm

load_dotenv(override=True)
openai = OpenAI()

DOCS = ROOT / "plan" / "docs"
documents = []  # the lesson's homemade loader
for folder in sorted(DOCS.iterdir()):
    if folder.is_dir():
        for file in sorted(folder.rglob("*.md")):
            documents.append({"type": folder.name, "source": file.relative_to(ROOT).as_posix(),
                              "text": file.read_text(encoding="utf-8")})
print(f"Loaded {len(documents)} documents")

results = []
for document in tqdm(documents):
    results.extend(chunk_document(completion, document))

chroma = PersistentClient(path=str(ROOT / "vector_db"))  # git leaves vector_db folders out
if "wwe_pro" in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection("wwe_pro")
collection = chroma.get_or_create_collection("wwe_pro")
texts = [result.page_content for result in results]
vectors = [e.embedding for e in openai.embeddings.create(model="text-embedding-3-large", input=texts).data]
collection.add(ids=[str(i) for i in range(len(texts))], embeddings=vectors, documents=texts,
               metadatas=[result.metadata for result in results])
print(f"Vectorstore created with {collection.count()} chunks")

history = [{"role": "user", "content": "Who won the first Money in the Bank ladder match?"},
           {"role": "assistant", "content": "Edge won it, at WrestleMania 21."}]
question = "When did he cash it in?"
query = rewrite_query(completion, question, history)
chunks = rerank(completion, query, retrieve(openai, collection, query))[:5]
context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
system = f"You answer questions about WWE history from these extracts. If they do not say, say so.\n\n{context}"
messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": question}]
print("Search for:", query)
print(completion(model="gpt-4.1-nano", messages=messages).choices[0].message.content)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.